In [2]:
!wget https://snap.stanford.edu/data/roadNet-CA.txt.gz
!gunzip roadNet-CA.txt.gz

--2026-06-01 05:47:47--  https://snap.stanford.edu/data/roadNet-CA.txt.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17892860 (17M) [application/x-gzip]
Saving to: ‘roadNet-CA.txt.gz’

roadNet-CA.txt.gz   100%[===================>]  17.06M  16.4MB/s    in 1.0s    

2026-06-01 05:47:48 (16.4 MB/s) - ‘roadNet-CA.txt.gz’ saved [17892860/17892860]



In [3]:
import numpy as np
from numba import cuda

# Read dataset
edges = []

with open("roadNet-CA.txt","r") as f:

    for line in f:

        if line.startswith("#"):
            continue

        u,v = map(int,line.split())
        edges.append((u,v))

edges = np.array(edges,dtype=np.int32)

print("Total Edges:",len(edges))

max_node = edges.max()+1

degree = np.zeros(max_node,dtype=np.int32)

# GPU kernel
@cuda.jit
def compute_degree(edges,degree):

    idx = cuda.grid(1)

    if idx < edges.shape[0]:

        u = edges[idx][0]
        v = edges[idx][1]

        cuda.atomic.add(degree,u,1)
        cuda.atomic.add(degree,v,1)

threads = 256
blocks = (len(edges)+threads-1)//threads

compute_degree[blocks,threads](edges,degree)

cuda.synchronize()

print("Nodes:",max_node)
print("Maximum Degree:",degree.max())
print("Average Degree:",degree.mean())

top_nodes = np.argsort(degree)[-10:]

print("\nTop Connected Nodes")

for node in top_nodes:
    print(node,degree[node])

Total Edges: 5533214
Nodes: 1971281
Maximum Degree: 24
Average Degree: 5.613825730578238

Top Connected Nodes
1052500 16
512565 16
1092460 16
309272 16
617103 16
291797 16
1795416 18
534751 20
521168 20
562818 24


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:934: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
